This folder is for checking data files and attempting to solve bugs in 3d-live-frame.py

In [1]:
import sys
import os
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import streamlit as st
import pandas as pd
import plotly.graph_objects as go
import pickle
import time


In [2]:
#source_path=r"C:\Users\asus\Desktop\zeynep calisma\neuroscience drosophila\sequential-inverse-kinematics\examples"
source_path=r".."
if source_path not in sys.path:
    sys.path.append(source_path)

from seqikpy.utils import load_file, save_file, calculate_body_size, dict_to_nparray_pose,from_sdf
from seqikpy.alignment import AlignPose, convert_from_df3dpp_to_dict, convert_from_anipose_to_dict
from seqikpy.kinematic_chain import KinematicChainSeq, KinematicChainGeneric
from seqikpy.leg_inverse_kinematics import LegInvKinSeq, LegInvKinGeneric
from seqikpy.visualization import plot_3d_points, animate_3d_points,generate_color_map
from seqikpy.body_config import neuromechfly_body_config
from seqikpy.head_inverse_kinematics import HeadInverseKinematics


In [3]:
# Set up the constant variables
leg_joint_angle_names = [
    "ThC_yaw",
    "ThC_pitch",
    "ThC_roll",
    "CTr_pitch",
    "CTr_roll",
    "FTi_pitch",
    "TiTa_pitch",
]
legs_to_align= ["RF", "RM", "RH", "LF", "LM", "LH"]

#loading processed sample data
#import_folder=Path(r"C:\Users\asus\Desktop\zeynep calisma\neuroscience drosophila\sequential-inverse-kinematics\data\inverse-kinematics-processed-data-for-comparison\Visualization-data")
import_folder=Path(r"..\..\data\inverse-kinematics-processed-data-for-comparison\Visualization-data")

#inverse kinematic outputs    
locomotion_leg_joint_angles=load_file(import_folder/"locomotion_leg_joint_angles_processed_300f.pkl")
locomotion_forward_kinematics=load_file(import_folder/"locomotion_forward_kinematics_processed_300f.pkl")

grooming_leg_joint_angles=load_file(import_folder/"grooming_leg_joint_angles_processed_300f.pkl")
grooming_forward_kinematics=load_file(import_folder/"grooming_forward_kinematics_processed_300f.pkl")

#original aligned posture
aligned_pos_locomotion=load_file(import_folder/"locomotion_aligned_pos_300f.h5")
aligned_pos_grooming=load_file(import_folder/"grooming_aligned_pos_300f.h5")
print("Sample files succesfully imported")


Sample files succesfully imported


In [27]:
#Bu kısımda aligned_pos objesinin hangi bacakları aldığına bakıp template'ı ona göre tanımla.

In [4]:
_TEMPLATE_NMF_LOCOMOTION = {
    "RF_Coxa": np.array([0.35, -0.27, 0.400]),
    "RF_Femur": np.array([0.35, -0.27, -0.025]),
    "RF_Tibia": np.array([0.35, -0.27, -0.731]),
    "RF_Tarsus": np.array([0.35, -0.27, -1.249]),
    "RF_Claw": np.array([0.35, -0.27, -1.912]),
    "LF_Coxa": np.array([0.35, 0.27, 0.400]),
    "LF_Femur": np.array([0.35, 0.27, -0.025]),
    "LF_Tibia": np.array([0.35, 0.27, -0.731]),
    "LF_Tarsus": np.array([0.35, 0.27, -1.249]),
    "LF_Claw": np.array([0.35, 0.27, -1.912]),
    "RM_Coxa": np.array([0, -0.125, 0]),
    "RM_Femur": np.array([0, -0.125, -0.182]),
    "RM_Tibia": np.array([0, -0.125, -0.965]),
    "RM_Tarsus": np.array([0, -0.125, -1.633]),
    "RM_Claw": np.array([0, -0.125, -2.328]),
    "LM_Coxa": np.array([0, 0.125, 0]),
    "LM_Femur": np.array([0, 0.125, -0.182]),
    "LM_Tibia": np.array([0, 0.125, -0.965]),
    "LM_Tarsus": np.array([0, 0.125, -1.633]),
    "LM_Claw": np.array([0, 0.125, -2.328]),
    "RH_Coxa": np.array([-0.215, -0.087, -0.073]),
    "RH_Femur": np.array([-0.215, -0.087, -0.272]),
    "RH_Tibia": np.array([-0.215, -0.087, -1.108]),
    "RH_Tarsus": np.array([-0.215, -0.087, -1.793]),
    "RH_Claw": np.array([-0.215, -0.087, -2.588]),
    "LH_Coxa": np.array([-0.215, 0.087, -0.073]),
    "LH_Femur": np.array([-0.215, 0.087, -0.272]),
    "LH_Tibia": np.array([-0.215, 0.087, -1.108]),
    "LH_Tarsus": np.array([-0.215, 0.087, -1.793]),
    "LH_Claw": np.array([-0.215, 0.087, -2.588]),
}

In [43]:
for i in list(aligned_pos_locomotion.keys()):
    print(i[:2:])

RF
RM
RH
LF
LM
LH


In [44]:
list(i[:2:] for i in list(aligned_pos_locomotion.keys()))

['RF', 'RM', 'RH', 'LF', 'LM', 'LH']

In [ ]:
def getTemplate(aligned_pos):
    _TEMPLATE_NMF_GENERAL = {
        "RF_Coxa": np.array([0.35, -0.27, 0.400]),
        "RF_Femur": np.array([0.35, -0.27, -0.025]),
        "RF_Tibia": np.array([0.35, -0.27, -0.731]),
        "RF_Tarsus": np.array([0.35, -0.27, -1.249]),
        "RF_Claw": np.array([0.35, -0.27, -1.912]),
        "LF_Coxa": np.array([0.35, 0.27, 0.400]),
        "LF_Femur": np.array([0.35, 0.27, -0.025]),
        "LF_Tibia": np.array([0.35, 0.27, -0.731]),
        "LF_Tarsus": np.array([0.35, 0.27, -1.249]),
        "LF_Claw": np.array([0.35, 0.27, -1.912]),
        "RM_Coxa": np.array([0, -0.125, 0]),
        "RM_Femur": np.array([0, -0.125, -0.182]),
        "RM_Tibia": np.array([0, -0.125, -0.965]),
        "RM_Tarsus": np.array([0, -0.125, -1.633]),
        "RM_Claw": np.array([0, -0.125, -2.328]),
        "LM_Coxa": np.array([0, 0.125, 0]),
        "LM_Femur": np.array([0, 0.125, -0.182]),
        "LM_Tibia": np.array([0, 0.125, -0.965]),
        "LM_Tarsus": np.array([0, 0.125, -1.633]),
        "LM_Claw": np.array([0, 0.125, -2.328]),
        "RH_Coxa": np.array([-0.215, -0.087, -0.073]),
        "RH_Femur": np.array([-0.215, -0.087, -0.272]),
        "RH_Tibia": np.array([-0.215, -0.087, -1.108]),
        "RH_Tarsus": np.array([-0.215, -0.087, -1.793]),
        "RH_Claw": np.array([-0.215, -0.087, -2.588]),
        "LH_Coxa": np.array([-0.215, 0.087, -0.073]),
        "LH_Femur": np.array([-0.215, 0.087, -0.272]),
        "LH_Tibia": np.array([-0.215, 0.087, -1.108]),
        "LH_Tarsus": np.array([-0.215, 0.087, -1.793]),
        "LH_Claw": np.array([-0.215, 0.087, -2.588]),
    }
    
    pose_list=list(i[:2:] for i in list(aligned_pos.keys()))
    pose_leg_list=[]
    for i in pose_list:
        if i in legs_to_align:
            pose_leg_list.append(i)
    pose_template=dict()

    for joint,location in _TEMPLATE_NMF_GENERAL.items():
        if joint[:2:] in pose_leg_list:
            pose_template[joint]=location
    return pose_template



In [6]:
body_config = neuromechfly_body_config.deepcopy()

In [7]:
body_config.template=getTemplate(aligned_pos_grooming)

In [8]:
def getInitAngleRad(aligned_pos):
    _INITIAL_ANGLES_RAD = {
        "RF": {
            # Base ThC yaw pitch CTr pitch
            "stage_1": np.array([0.0, 0.45, -0.07, -2.14]),
            # Base ThC yaw pitch roll CTr pitch CTr roll
            "stage_2": np.array([0.0, 0.45, -0.07, -0.32, -2.14, 1.4]),
            # Base ThC yaw pitch roll CTr pitch CTr roll FTi pitch
            "stage_3": np.array([0.0, 0.45, -0.07, -0.32, -2.14, -1.25, 1.48, 0.0]),
            # Base ThC yaw pitch roll CTr pitch CTr roll FTi pitch TiTa pitch
            "stage_4": np.array([0.0, 0.45, -0.07, -0.32, -2.14, -1.25, 1.48, 0.0, 0.0]),
        },
        "LF": {
            "stage_1": np.array([0.0, -0.45, -0.07, -2.14]),
            "stage_2": np.array([0.0, -0.45, -0.07, 0.32, -2.14, 1.4]),
            "stage_3": np.array([0.0, -0.45, -0.07, 0.32, -2.14, 1.25, 1.48, 0.0]),
            "stage_4": np.array([0.0, -0.45, -0.07, 0.32, -2.14, 1.25, 1.48, 0.0, 0.0]),
        },
        "RM": {
            "stage_1": np.array([0.0, 0.45, 0.37, -2.14]),
            "stage_2": np.array([0.0, 0.45, 0.37, -0.32, -2.14, 1.4]),
            "stage_3": np.array([0.0, 0.45, 0.37, -0.32, -2.14, -1.25, 1.48, 0.0]),
            "stage_4": np.array([0.0, 0.45, 0.37, -0.32, -2.14, -1.25, 1.48, 0.0, 0.0]),
        },
        "LM": {
            "stage_1": np.array([0.0, -0.45, 0.37, -2.14]),
            "stage_2": np.array([0.0, -0.45, 0.37, 0.32, -2.14, 1.4]),
            "stage_3": np.array([0.0, -0.45, 0.37, 0.32, -2.14, 1.25, 1.48, 0.0]),
            "stage_4": np.array([0.0, -0.45, 0.37, 0.32, -2.14, 1.25, 1.48, 0.0, 0.0]),
        },
        "RH": {
            "stage_1": np.array([0.0, 0.45, 0.07, -2.14]),
            "stage_2": np.array([0.0, 0.45, 0.07, -0.32, -2.14, 1.4]),
            "stage_3": np.array([0.0, 0.45, 0.07, -0.32, -2.14, -1.25, 1.48, 0.0]),
            "stage_4": np.array([0.0, 0.45, 0.07, -0.32, -2.14, -1.25, 1.48, 0.0, 0.0]),
        },
        "LH": {
            "stage_1": np.array([0.0, -0.45, 0.07, -2.14]),
            "stage_2": np.array([0.0, -0.45, 0.07, 0.32, -2.14, 1.4]),
            "stage_3": np.array([0.0, -0.45, 0.07, 0.32, -2.14, 1.25, 1.48, 0.0]),
            "stage_4": np.array([0.0, -0.45, 0.07, 0.32, -2.14, 1.25, 1.48, 0.0, 0.0]),
        },
    }
    pose_leg_list=list(i[:2:] for i in list(aligned_pos.keys()))
    pose_init_angles_rad=dict()

    for leg,angles in _INITIAL_ANGLES_RAD.items():
        if leg in pose_leg_list:
            pose_init_angles_rad[leg]=angles

    return pose_init_angles_rad

    

In [9]:
body_config.initial_angles_rad = getInitAngleRad(aligned_pos_grooming)

In [10]:
def getBounds(aligned_pos):
    _BOUNDS_DEG = {
        "RF_ThC_yaw": (-180, 180),
        "RF_ThC_pitch": (-90, 90),
        "RF_ThC_roll": (-180, 180),
        "RF_CTr_pitch": (-180, 180),
        "RF_FTi_pitch": (-180, 180),
        "RF_CTr_roll": (-180, 180),
        "RF_TiTa_pitch": (-180, 0),
        "RM_ThC_yaw": (-50, 50),
        "RM_ThC_pitch": (-180, 180),
        "RM_ThC_roll": (-180, 0),
        "RM_CTr_pitch": (-180, 180),
        "RM_FTi_pitch": (-180, 180),
        "RM_CTr_roll": (-180, 180),
        "RM_TiTa_pitch": (-180, 0),
        "RH_ThC_yaw": (-50, 50),
        "RH_ThC_pitch": (-50, 50),
        "RH_ThC_roll": (-180, 0),
        "RH_CTr_pitch": (-180, 0),
        "RH_FTi_pitch": (-180, 180),
        "RH_CTr_roll": (-180, 180),
        "RH_TiTa_pitch": (-180, 0),
        "LF_ThC_yaw": (-180, 180),
        "LF_ThC_pitch": (-90, 90),
        "LF_ThC_roll": (-180, 180),
        "LF_CTr_pitch": (-180, 180),
        "LF_FTi_pitch": (-180, 180),
        "LF_CTr_roll": (-180, 180),
        "LF_TiTa_pitch": (-180, 0),
        "LM_ThC_yaw": (-50, 50),
        "LM_ThC_pitch": (-180, 180),
        "LM_ThC_roll": (0, 180),
        "LM_CTr_pitch": (-180, 180),
        "LM_FTi_pitch": (-180, 180),
        "LM_CTr_roll": (-180, 180),
        "LM_TiTa_pitch": (-180, 0),
        "LH_ThC_yaw": (-50, 50),
        "LH_ThC_pitch": (-50, 50),
        "LH_ThC_roll": (0, 180),
        "LH_CTr_pitch": (-180, 0),
        "LH_FTi_pitch": (-180, 180),
        "LH_CTr_roll": (-180, 180),
        "LH_TiTa_pitch": (-180, 0),
    }
    pose_leg_list=list(i[:2:] for i in list(aligned_pos.keys()))
    pose_bounds_deg=dict()

    for jointmov,bounds in _BOUNDS_DEG.items():
        if jointmov[:2:] in pose_leg_list:
            pose_bounds_deg[jointmov]=bounds

    return pose_bounds_deg




In [11]:
body_config.set_dof_bounds_in_deg(getBounds(aligned_pos_grooming))

In [56]:
aligned_pos_grooming.keys()

dict_keys(['R_head', 'RF_leg', 'L_head', 'LF_leg', 'Neck'])

In [12]:
def clearPose(aligned_pos):# for leaving legs only
    leg_names=[
        "RF_leg",
        "RM_leg",
        "RH_leg",
        "LF_leg",
        "LM_leg",
        "LH_leg",
        
    ]
    clear_pose=dict()
    for limb, data in aligned_pos.items():
        if limb in leg_names:
            clear_pose[limb]=data
    return clear_pose

In [13]:
aligned_pos_grooming_clear=clearPose(aligned_pos_grooming)

In [14]:
aligned_pos_grooming_clear.keys()

dict_keys(['RF_leg', 'LF_leg'])

In [ ]:
#NOTE: Only legs defined in template. make sure it works without thoraxMid, antennas, wings and neck.
#NOTE2: It works :)

In [ ]:
# Initialize the necessary classes
kin_chain = KinematicChainSeq(
    bounds_dof=body_config.dof_bounds_rad,
    body_size=calculate_body_size(
        body_config.template,
        legs_list=["RF"] #no need to give all
    ),
    legs_list=legs_to_align,
)

class_seq_ik = LegInvKinSeq(
    aligned_pos=aligned_pos_grooming,
    kinematic_chain_class=kin_chain,
    initial_angles=body_config.initial_angles_rad,
)

In [78]:
leg_joint_angles, forward_kinematicsOriginal = class_seq_ik.run_ik_and_fk(
    hide_progress_bar=False
)

RF stage 1:   0%|          | 0/300 [00:00<?, ?it/s]

RF stage 4: 100%|██████████| 300/300 [01:38<00:00,  3.06it/s]


### Forward Kinematics Calculation try

In [79]:
leg_joint_angles

{'Angle_RF_ThC_yaw': array([ 0.31451559,  0.28191389,  0.23992888,  0.19739981,  0.16246689,
         0.14002526,  0.13148819,  0.13647219,  0.1532652 ,  0.18116461,
         0.21857762,  0.26351963,  0.30994425,  0.34710398,  0.36547759,
         0.36255361,  0.34022678,  0.30105267,  0.24691065,  0.19123672,
         0.15588345,  0.15810851,  0.1972832 ,  0.25478634,  0.30884556,
         0.34644153,  0.36286439,  0.35885364,  0.33822241,  0.30627108,
         0.26693351,  0.2251813 ,  0.1872468 ,  0.16063441,  0.15151904,
         0.16258495,  0.19052774,  0.22801747,  0.26515248,  0.29359794,
         0.31007596,  0.31372316,  0.30505846,  0.28587498,  0.26001966,
         0.23201072,  0.20836675,  0.19075271,  0.18067619,  0.17890416,
         0.18010114,  0.17589983,  0.16229107,  0.14370561,  0.12682704,
         0.11533124,  0.10934299,  0.1060673 ,  0.10132241,  0.09297113,
         0.07953634,  0.06089405,  0.03966343,  0.02160693,  0.01128497,
         0.01039836,  0.0175653

In [31]:
# Initialize the necessary classes
kin_chain = KinematicChainSeq(
    bounds_dof=body_config.dof_bounds_rad,
    body_size=calculate_body_size(
        body_config.template,
        legs_list=["RF"] #no need to give all
    ),
    legs_list=legs_to_align,
)

class_seq_ik = LegInvKinSeq(
    aligned_pos=aligned_pos_grooming,
    kinematic_chain_class=kin_chain,
    initial_angles=body_config.initial_angles_rad,
)

In [53]:
leg_joint_angles_for_fk= np.array([
    leg_joint_angles["Angle_RF_ThC_yaw"][0],
    leg_joint_angles["Angle_RF_ThC_pitch"][0],
    leg_joint_angles["Angle_RF_ThC_roll"][0],
    leg_joint_angles["Angle_RF_CTr_pitch"][0],
    leg_joint_angles["Angle_RF_CTr_roll"][0],
    leg_joint_angles["Angle_RF_FTi_pitch"][0],
    leg_joint_angles["Angle_RF_TiTa_pitch"][0]
    ])

In [58]:
leg_joint_angles_for_fk

array([ 0.31451559, -0.07141676, -0.20330969, -1.92322878, -1.30802191,
        1.60869755, -1.19906266])

In [73]:
from forward_kinematics_from_angles import calculate_fk_from_seq_angles

In [75]:
forward_kinematics = calculate_fk_from_seq_angles(
    joint_angles=leg_joint_angles,
    kinematic_chain_seq=kin_chain,
    legs=["RF"],
    origins=None,
    anatomical_points_only=True,
)

rf_xyz = forward_kinematics["RF_leg"]

In [80]:
forward_kinematicsOriginal

{'RF_leg': array([[[ 0.33      , -0.17      ,  1.07      ],
         [ 0.33      , -0.17      ,  1.07      ],
         [ 0.33      , -0.17      ,  1.07      ],
         ...,
         [ 0.99024177, -0.25560697,  0.90067881],
         [ 1.12774516,  0.2427649 ,  0.93296344],
         [ 1.74943151,  0.30721725,  1.15414429]],
 
        [[ 0.33      , -0.17      ,  1.07      ],
         [ 0.33      , -0.17      ,  1.07      ],
         [ 0.33      , -0.17      ,  1.07      ],
         ...,
         [ 0.95342804, -0.30653406,  0.90007853],
         [ 1.1462622 ,  0.17420353,  0.89456895],
         [ 1.77635901,  0.22632351,  1.09414481]],
 
        [[ 0.33      , -0.17      ,  1.07      ],
         [ 0.33      , -0.17      ,  1.07      ],
         [ 0.33      , -0.17      ,  1.07      ],
         ...,
         [ 0.88042237, -0.38979871,  0.91813634],
         [ 1.14061727,  0.05619792,  0.87678878],
         [ 1.77643976,  0.08215234,  1.06286707]],
 
        ...,
 
        [[ 0.33      , -

In [77]:
forward_kinematics

{'RF_leg': array([[[ 0.        ,  0.        ,  0.        ],
         [ 0.03032633,  0.1311411 , -0.40312197],
         [ 0.66024177, -0.08560697, -0.16932119],
         [ 0.79774516,  0.4127649 , -0.13703656],
         [ 1.41943151,  0.47721725,  0.08414429]],
 
        [[ 0.        ,  0.        ,  0.        ],
         [ 0.00958193,  0.11820261, -0.40811926],
         [ 0.62342804, -0.13653406, -0.16992147],
         [ 0.8162622 ,  0.34420353, -0.17543105],
         [ 1.44635901,  0.39632351,  0.02414481]],
 
        [[ 0.        ,  0.        ,  0.        ],
         [-0.02212124,  0.10085736, -0.41226623],
         [ 0.55042237, -0.21979871, -0.15186366],
         [ 0.81061727,  0.22619792, -0.19321122],
         [ 1.44643976,  0.25215234, -0.00713293]],
 
        ...,
 
        [[ 0.        ,  0.        ,  0.        ],
         [-0.18406646,  0.02822555, -0.38203122],
         [ 0.42542762, -0.18869698, -0.09936962],
         [ 0.58388356,  0.02444568, -0.54410091],
         [ 1.111